# Titanic Survival Prediction

**Objective:** Predict passenger survival on the Titanic using multiple classification models.

**Dataset:** [Kaggle Titanic Training Data](https://www.kaggle.com/c/titanic/data) (891 passengers, 12 features)

**Models compared:**
1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Bagging Classifier
5. AdaBoost
6. Gradient Boosting
7. K-Nearest Neighbors

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    BaggingClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
)

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

RANDOM_STATE = 42

## 2. Load & Explore Data

In [ ]:
df = pd.read_csv("titanic-training-data.csv")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("Missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(x="Survived", data=df, ax=axes[0, 0], hue="Survived", palette="Set2", legend=False)
axes[0, 0].set_title("Survival Count")
axes[0, 0].set_xticklabels(["Died", "Survived"])

sns.countplot(x="Pclass", hue="Survived", data=df, ax=axes[0, 1], palette="Set2")
axes[0, 1].set_title("Survival by Passenger Class")

sns.countplot(x="Sex", hue="Survived", data=df, ax=axes[1, 0], palette="Set2")
axes[1, 0].set_title("Survival by Sex")

sns.countplot(x="Embarked", hue="Survived", data=df, ax=axes[1, 1], palette="Set2")
axes[1, 1].set_title("Survival by Embarkation Port")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["Age"].dropna(), bins=30, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Age Distribution")

sns.boxplot(x="Pclass", y="Age", data=df, ax=axes[1], hue="Pclass", palette="Set2", legend=False)
axes[1].set_title("Age by Passenger Class")

plt.tight_layout()
plt.show()

In [ ]:
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

### Key EDA Observations
- **Class matters:** 1st class passengers had much higher survival rates.
- **Gender matters:** Females survived at a significantly higher rate than males.
- **Age:** Children had better survival chances; Age has ~20% missing values.
- **Embarked:** Cherbourg (C) passengers had slightly higher survival rates.
- **Cabin:** 77% missing — too sparse to use directly, but we can extract the deck letter.

## 4. Data Preprocessing

### 4.1 Handle Missing Values

In [ ]:
# Age: fill with median by Pclass (more accurate than global mean)
df["Age"] = df.groupby("Pclass")["Age"].transform(lambda x: x.fillna(x.median()))

# Embarked: fill 2 missing values with mode ("S" = Southampton)
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Cabin: extract deck letter (first character), fill missing as "Unknown"
df["Deck"] = df["Cabin"].apply(lambda x: x[0] if pd.notna(x) else "Unknown")

print("Missing values after handling:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("(No missing values remain)" if df.isnull().sum().sum() == 0 else "")

### 4.2 Feature Engineering

In [ ]:
# Family size = siblings/spouses + parents/children + self
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

# Drop columns that won't help prediction
drop_cols = ["PassengerId", "Name", "Ticket", "Cabin"]
df = df.drop(columns=drop_cols)

# One-hot encode categorical features
df = pd.get_dummies(df, columns=["Sex", "Embarked", "Deck"], drop_first=True)

print(f"Final feature set: {df.shape[1] - 1} features")
df.head()

### 4.3 Train/Test Split & Feature Scaling

In [ ]:
X = df.drop(columns=["Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

# Scale features (important for Logistic Regression and KNN)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"Survival rate — Train: {y_train.mean():.2%}, Test: {y_test.mean():.2%}")

## 5. Model Training & Evaluation

We train 7 classification models and compare them using accuracy, cross-validation score, and ROC-AUC.

In [ ]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    """Train, evaluate, and return results for a single model."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1] if hasattr(model, "predict_proba") else None

    train_acc = model.score(X_tr, y_tr)
    test_acc = accuracy_score(y_te, y_pred)
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=5, scoring="accuracy")
    roc_auc = roc_auc_score(y_te, y_prob) if y_prob is not None else None

    print(f"\n{'='*50}")
    print(f" {name}")
    print(f"{'='*50}")
    print(f"  Train Accuracy:  {train_acc:.4f}")
    print(f"  Test Accuracy:   {test_acc:.4f}")
    print(f"  CV Accuracy:     {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    if roc_auc:
        print(f"  ROC-AUC:         {roc_auc:.4f}")
    print(f"\n{classification_report(y_te, y_pred, target_names=['Died', 'Survived'])}")

    return {
        "Model": name,
        "Train Acc": train_acc,
        "Test Acc": test_acc,
        "CV Mean": cv_scores.mean(),
        "CV Std": cv_scores.std(),
        "ROC-AUC": roc_auc,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }

### 5.1 Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
results_lr = evaluate_model("Logistic Regression", lr, X_train_scaled, X_test_scaled, y_train, y_test)

### 5.2 Decision Tree

In [ ]:
dt = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)
results_dt = evaluate_model("Decision Tree", dt, X_train, X_test, y_train, y_test)

### 5.3 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=RANDOM_STATE)
results_rf = evaluate_model("Random Forest", rf, X_train, X_test, y_train, y_test)

### 5.4 Bagging Classifier

In [ ]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=5),
    n_estimators=20,
    random_state=RANDOM_STATE,
)
results_bag = evaluate_model("Bagging Classifier", bag, X_train, X_test, y_train, y_test)

### 5.5 AdaBoost

In [ ]:
ada = AdaBoostClassifier(n_estimators=100, random_state=RANDOM_STATE)
results_ada = evaluate_model("AdaBoost", ada, X_train, X_test, y_train, y_test)

### 5.6 Gradient Boosting

In [ ]:
gbc = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=RANDOM_STATE)
results_gbc = evaluate_model("Gradient Boosting", gbc, X_train, X_test, y_train, y_test)

### 5.7 K-Nearest Neighbors

In [ ]:
knn = KNeighborsClassifier(n_neighbors=7)
results_knn = evaluate_model("KNN (k=7)", knn, X_train_scaled, X_test_scaled, y_train, y_test)

## 6. Model Comparison

In [ ]:
all_results = [results_lr, results_dt, results_rf, results_bag, results_ada, results_gbc, results_knn]

comparison = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ["y_pred", "y_prob"]}
    for r in all_results
]).sort_values("Test Acc", ascending=False).reset_index(drop=True)

comparison.style.format({
    "Train Acc": "{:.4f}",
    "Test Acc": "{:.4f}",
    "CV Mean": "{:.4f}",
    "CV Std": "{:.4f}",
    "ROC-AUC": "{:.4f}",
}).highlight_max(subset=["Test Acc", "CV Mean", "ROC-AUC"], color="lightgreen")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy comparison
models = [r["Model"] for r in all_results]
train_accs = [r["Train Acc"] for r in all_results]
test_accs = [r["Test Acc"] for r in all_results]

x = np.arange(len(models))
width = 0.35
axes[0].bar(x - width/2, train_accs, width, label="Train", color="steelblue")
axes[0].bar(x + width/2, test_accs, width, label="Test", color="coral")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Train vs Test Accuracy")
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=45, ha="right")
axes[0].legend()
axes[0].set_ylim(0.6, 1.0)

# ROC Curves
for r in all_results:
    if r["y_prob"] is not None:
        fpr, tpr, _ = roc_curve(y_test, r["y_prob"])
        axes[1].plot(fpr, tpr, label=f"{r['Model']} (AUC={r['ROC-AUC']:.3f})")

axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curves")
axes[1].legend(loc="lower right", fontsize=8)

plt.tight_layout()
plt.show()

## 7. Confusion Matrices (Top 3 Models)

In [ ]:
top_3 = sorted(all_results, key=lambda r: r["Test Acc"], reverse=True)[:3]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, r in zip(axes, top_3):
    cm = confusion_matrix(y_test, r["y_pred"])
    sns.heatmap(
        pd.DataFrame(cm, index=["Died", "Survived"], columns=["Died", "Survived"]),
        annot=True, fmt="d", cmap="Blues", ax=ax
    )
    ax.set_title(f"{r['Model']}\nAccuracy: {r['Test Acc']:.4f}")
    ax.set_ylabel("Actual")
    ax.set_xlabel("Predicted")

plt.tight_layout()
plt.show()

## 8. Feature Importance (Best Tree-Based Model)

In [ ]:
best_tree = max(
    [("Random Forest", rf), ("Gradient Boosting", gbc)],
    key=lambda x: x[1].score(X_test, y_test)
)

importances = pd.Series(best_tree[1].feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 8))
importances.plot(kind="barh", color="steelblue")
plt.title(f"Feature Importance ({best_tree[0]})")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 9. Conclusion

### Summary
- **7 models** were trained and compared on the Titanic survival dataset.
- **Ensemble methods** (Random Forest, Gradient Boosting) generally outperformed simpler models.
- **Key predictive features:** Sex (male), Fare, Age, and Passenger Class were the strongest predictors of survival.
- **Cross-validation** confirmed that results are stable and not due to a lucky train/test split.

### What Helped
- Filling Age by class-median instead of global mean preserved class-specific age distributions.
- Feature scaling significantly improved Logistic Regression and KNN performance.
- Extracting the Deck letter from Cabin added signal despite 77% missing values.
- Adding FamilySize and IsAlone features captured group survival dynamics.

### Possible Next Steps
- Hyperparameter tuning with `GridSearchCV` or `RandomizedSearchCV`.
- Stacking or blending top-performing models.
- Extract titles from passenger names (Mr, Mrs, Miss, Master) as a feature.
- Submit predictions on the Kaggle test set.